## Self-reflection topology demonstration

In this notebook we present the capabilities of the self-reflection multi-agent topology. In this setup, an agent provides an answer to a provided prompt and then provides feedback on its own output (self-reflection). To illustrate the capabilities of this agent topology we will load a sample dataset (*California Housing dataset*) and write a simple neural network to predict the median house value. The self-reflection feedback will then be used to to update the output of the agent. 

#### MAS setup

In [3]:
import sys
import os
import json
from pathlib import Path
from IPython.display import Markdown, display

# If the notebook lives in examples/, make sure src/ is importable
sys.path.insert(0, str(Path.cwd().parent / "src"))

from mas_topologies import TopologyAgent, PromptSet



**API setup:**

In [ ]:
os.environ["OPENAI_API_KEY"] = "..."
os.environ["ANTHROPIC_API_KEY"] = "..."

In [ ]:
def build_client(provider="auto"):
    if provider in ("openai", "auto") and os.environ.get("OPENAI_API_KEY"):
        from autogen_ext.models.openai import OpenAIChatCompletionClient
        model = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
        client = OpenAIChatCompletionClient(model=model, api_key=os.environ["OPENAI_API_KEY"])
        return client, f"openai/{model}"

    if provider in ("anthropic", "auto") and os.environ.get("ANTHROPIC_API_KEY"):
        from autogen_ext.models.anthropic import AnthropicChatCompletionClient
        model = os.environ.get("ANTHROPIC_MODEL", "claude-haiku-4-5")
        client = AnthropicChatCompletionClient(model=model, api_key=os.environ["ANTHROPIC_API_KEY"])
        return client, f"anthropic/{model}"

    raise EnvironmentError(
        "No API key found. Export OPENAI_API_KEY or ANTHROPIC_API_KEY before running this notebook."
    )

MODEL_CLIENT, PROVIDER_LABEL = build_client("auto")
print(f"Provider : {PROVIDER_LABEL}")

Provider : openai/gpt-4o-mini


#### Agent prompt setup:

In [ ]:
SYSTEM_PROMPT = """You are an expert machine learning engineer who writes PyTorch models
and then self-reflects on your own design.

Your job:
1. Propose a regression model for the California Housing dataset.
2. Explain the architecture and why it fits the task.
3. Return complete, executable Python code.
4. Then reflect on your own answer and improve it if needed."""

REFLECTION_PROMPT = """Review the model proposal above.
Check:
- Tensor shape consistency
- Model capacity vs dataset size
- Correct regression output shape
- Appropriate regularisation and activation choices
- Whether the model returns raw logits / regression output correctly

If you find issues, produce a corrected full response.
If the design is already good, confirm why it is correct."""

PROMPT_SET = PromptSet(
    system_message=SYSTEM_PROMPT,
    reflection_prompt=REFLECTION_PROMPT,
)

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

HOUSING_CONTEXT = """California Housing regression task:
- Input: numeric housing features (e.g. median income, house age, bedrooms, population)
- Output: predicted median house value
- Target: build a simple PyTorch regression model with sensible preprocessing guidance
- Requirements: return a model ready for training, explain the architecture, and include a short hyperparameter recommendation.
"""

agent = TopologyAgent(
    model_client=MODEL_CLIENT,
    topology="self_reflection",
    prompts=PROMPT_SET,
    log_dir=LOG_DIR,
)

print("Self-reflection agent ready")
print(HOUSING_CONTEXT)

#### Dataset setup

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import pandas as pd

# Load the dataset
housing = fetch_california_housing(as_frame=True)

# Convert to a DataFrame for easy inspection
df = housing.frame
print("Dataset shape:", df.shape)
print("Features:", housing.feature_names)
display(df.head())

# Separate features and target
X = df[housing.feature_names]
y = df["MedHouseVal"]

# Train / validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train shape: {X_train.shape}, Validation shape: {X_val.shape}")